In [17]:
import pandas as pd
import re
import ast

In [18]:
df = pd.read_csv(filepath_or_buffer="Data/Raw/zomato.csv",
                usecols=['name','address','online_order', 'book_table', 'rate', 'votes',
                'location', 'rest_type', 'cuisines',
                'approx_cost(for two people)', 'reviews_list'])
df.head()

,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_cost(for two people),reviews_list
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ..."
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din..."
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ..."
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,Banashankari,Quick Bites,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper..."
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ..."


In [19]:
df.dtypes

address                        object
name                           object
online_order                   object
book_table                     object
rate                           object
votes                           int64
location                       object
rest_type                      object
cuisines                       object
approx_cost(for two people)    object
reviews_list                   object
dtype: object

In [20]:
def extract_rating(rating:str) -> float:
    ''' 
    This function takes the rating in the dataset (which is a string) and it converts the rating to a float number
    '''
    # Check if the rating matches the format digit.digit/5
    if re.match(r'^\d+\.\d+/5$', str(rating)):
        return float(rating.split('/')[0])
    else:
        return None

In [21]:
# Initial data situation
print(f'Number of rows with initially NULL values {df['rate'].isnull().sum()}')
print(f"Number of rows with initially NEW value: {len(df[df['rate'] == 'NEW'])}")

# Performing the cleaning
print(f'Type for the rate before processing : {type(df['rate'][0])}')
df['Rating'] = df['rate'].apply(lambda x: extract_rating(x))
print(f'Type for the rate after processing : {type(df["Rating"][0])}')

# Removing all the null values that are now present
print(f"Shape of the raw dataframe is {df.shape}")
df.dropna(subset=['Rating'], axis=0, inplace=True) #dropping all the rows where the rating is null (These were initially empty or NEW)
print(f"Shape of the dataframe after dropping null values is {df.shape}")
df.drop(['rate'],axis=1,inplace=True)

Number of rows with initially NULL values 7775
Number of rows with initially NEW value: 2208
Type for the rate before processing : <class 'str'>
Type for the rate after processing : <class 'numpy.float64'>
Shape of the raw dataframe is (51717, 12)
Shape of the dataframe after dropping null values is (21288, 12)


In [22]:
df.head()

,address,name,online_order,book_table,votes,location,rest_type,cuisines,approx_cost(for two people),reviews_list,Rating
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",4.1
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",4.1
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",3.8
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,88,Banashankari,Quick Bites,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",3.7
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",3.8


In [23]:
def fix_reviews(review_list):
    filtered_reviews = []
    for rev_tuple in review_list:
        rating_str = rev_tuple[0]
        review = rev_tuple[1]
        if rating_str is not None and review is not None:
            # Extracting the rating
            rating = re.search(r'Rated ([0-9]+\.?[0-9]*)', rating_str)
            if rating:
                rating = float(rating.group(1))
            else:
                rating = None
            # Extracting the review text
            review_text = review.split('\n')[-1].strip()
            filtered_reviews.append((rating, review_text))
    return filtered_reviews

In [24]:
df['reviews'] = df['reviews_list'].apply(lambda x: fix_reviews(list(set(ast.literal_eval(x)))) if isinstance(x, str) else fix_reviews(list(set(x))))
df.drop(['reviews_list'],axis=1,inplace=True)

In [25]:
df.head()

,address,name,online_order,book_table,votes,location,rest_type,cuisines,approx_cost(for two people),Rating,reviews
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800,4.1,"[(4.0, Cheers), (5.0, subirmajumder85.wixsite...."
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800,4.1,"[(4.0, Will definitely visit again ??), (4.0, ..."
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800,3.8,"[(4.0, 5) over San churro cafe good....), (3.0..."
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,88,Banashankari,Quick Bites,"South Indian, North Indian",300,3.7,"[(4.5, Very good and Unlimited . especially ma..."
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600,3.8,"[(4.0, Pocket friendly : 4/5), (4.0, Very good..."


In [26]:
df['reviews'][0]

[(4.0, 'Cheers'),
 (5.0, 'subirmajumder85.wixsite.com'),
 (4.0,
  'The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'),
 (4.0, 'In the mocktails, recommend "Alice in Junoon". Do not miss it.'),
 (4.0,
  'A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.'),
 (5.0, 'Overall :5/5'),
 (4.0,
  'We went here on a weekend and one of us had the buffet while two of us took Ala Carte. Firstly the ambience and service of this place is great! The buffet had a lot of items and the good was good. We had a Pumpkin Halwa intm the dessert which was amazing. Must try! The kulchas are great here. Cheers!'),
 (2.0,
  'Its a res

In [27]:
print(f"Final dataframe size : {df.shape}")

Final dataframe size : (21288, 11)


In [28]:
df.isna().sum()

address                         0
name                            0
online_order                    0
book_table                      0
votes                           0
location                        0
rest_type                      85
cuisines                        4
approx_cost(for two people)    75
Rating                          0
reviews                         0
dtype: int64

In [29]:
df.dropna(subset=['rest_type'], axis=0, inplace=True)
df.dropna(subset=['cuisines'], axis=0, inplace=True)
df.dropna(subset=['approx_cost(for two people)'], axis=0, inplace=True)

In [30]:
def price_per_head(price: str) -> float:
    ''' 
    This function takes the approx cost for two people in the dataset (which is a string), 
    converts it to a float number, then divides by 2 to get the cost per head.
    '''
    if ',' in price:
        price = price.replace(',', '')  # Reassign the modified string to price
    return float(price) / 2  # Convert the cleaned price to float and divide by 2

In [31]:
# Dropping the null values
print(f"Number of rows initially with NULL Values : {df['approx_cost(for two people)'].isnull().sum()}")
print(f"Shape of the dataframe before dropping null values is {df.shape}")
df.dropna(subset=['approx_cost(for two people)'], axis=0, inplace= True)
print(f"Shape of the dataframe after dropping null values is {df.shape}")

# Computing the price per head
df['Price Per Head'] = df['approx_cost(for two people)'].apply(lambda x: price_per_head(x))
df.drop(['approx_cost(for two people)'],axis=1,inplace=True)

Number of rows initially with NULL Values : 0
Shape of the dataframe before dropping null values is (21124, 11)
Shape of the dataframe after dropping null values is (21124, 11)


In [32]:
df.head()

,address,name,online_order,book_table,votes,location,rest_type,cuisines,Rating,reviews,Price Per Head
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",4.1,"[(4.0, Cheers), (5.0, subirmajumder85.wixsite....",400.0
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",4.1,"[(4.0, Will definitely visit again ??), (4.0, ...",400.0
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",3.8,"[(4.0, 5) over San churro cafe good....), (3.0...",400.0
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,88,Banashankari,Quick Bites,"South Indian, North Indian",3.7,"[(4.5, Very good and Unlimited . especially ma...",150.0
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",3.8,"[(4.0, Pocket friendly : 4/5), (4.0, Very good...",300.0


In [35]:
# Performing train-test split of 80% train and 10% validation and 10% test
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2, random_state=42)
test, val = train_test_split(test, test_size=0.1, random_state=42)
print(f"Shape of the train dataframe is {train.shape}")
print(f"Shape of the test dataframe is {test.shape}")
print(f"Shape of the validation dataframe is {val.shape}")

Shape of the train dataframe is (16899, 11)
Shape of the test dataframe is (3802, 11)
Shape of the validation dataframe is (423, 11)


In [37]:
# saving the dataframes to csv files
train.to_csv('Data/Formatted/train.csv', index=False)
test.to_csv('Data/Formatted/test.csv', index=False)
val.to_csv('Data/Formatted/val.csv', index=False)